In [1]:
#import

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import time

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")


PyTorch version: 2.2.2
MPS available: True


In [2]:
#load
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = pd.read_csv('../data/processed/y_train_resampled.csv').squeeze()

X_val = np.load('../data/processed/X_val_scaled.npy')
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2062829, 71)
X_val shape: (423051, 71)
y_train shape: (2062829,)


In [3]:
#label encoding cause CNN does not work with strings
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

print("Classes and their encodings:")
for i, label in enumerate(le.classes_):
    print(f"  {i} → {label}")

Classes and their encodings:
  0 → BENIGN
  1 → Bot
  2 → DDoS
  3 → DoS GoldenEye
  4 → DoS Hulk
  5 → DoS Slowhttptest
  6 → DoS slowloris
  7 → FTP-Patator
  8 → Heartbleed
  9 → Infiltration
  10 → PortScan
  11 → SSH-Patator
  12 → Web Attack - Brute Force
  13 → Web Attack - Sql Injection
  14 → Web Attack - XSS


In [4]:
#create pytorch dataset

class NetworkFlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    # how many samples are in the dataset
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [5]:
train_dataset = NetworkFlowDataset(X_train, y_train_encoded)
val_dataset = NetworkFlowDataset(X_val, y_val_encoded)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 4029
Validation batches: 827


In [6]:
class CNN1D(nn.Module):
    def __init__(self, input_size, num_classes):
        super(CNN1D, self).__init__()
        
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
        conv_output_size = (input_size // 4) * 128
        
        self.fc1 = nn.Linear(conv_output_size, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = CNN1D(input_size=71, num_classes=15).to(device)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps
Model parameters: 586,127


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam, lr=0.001")

Loss function: CrossEntropyLoss
Optimizer: Adam, lr=0.001


In [9]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    model.train()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{epochs} — Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Epoch 1/10 — Loss: 0.1005, Accuracy: 96.02%
Epoch 2/10 — Loss: 0.0627, Accuracy: 97.24%
Epoch 3/10 — Loss: 0.0564, Accuracy: 97.49%
Epoch 4/10 — Loss: 0.0535, Accuracy: 97.69%
Epoch 5/10 — Loss: 0.0503, Accuracy: 97.84%
Epoch 6/10 — Loss: 0.0471, Accuracy: 98.00%
Epoch 7/10 — Loss: 0.0445, Accuracy: 98.09%
Epoch 8/10 — Loss: 0.0420, Accuracy: 98.18%
Epoch 9/10 — Loss: 0.0407, Accuracy: 98.22%
Epoch 10/10 — Loss: 0.0389, Accuracy: 98.29%


In [10]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

all_preds = le.inverse_transform(all_preds)
all_labels = le.inverse_transform(all_labels)

print(classification_report(all_labels, all_preds, digits=4))

                            precision    recall  f1-score   support

                    BENIGN     0.9959    0.9892    0.9926    339790
                       Bot     0.3603    0.8805    0.5114       293
                      DDoS     0.9994    0.9992    0.9993     19153
             DoS GoldenEye     0.9915    0.9838    0.9876      1540
                  DoS Hulk     0.9808    0.9966    0.9887     34427
          DoS Slowhttptest     0.9113    0.9866    0.9475       823
             DoS slowloris     0.9896    0.9919    0.9908       867
               FTP-Patator     0.9983    0.9975    0.9979      1187
                Heartbleed     0.6667    1.0000    0.8000         2
              Infiltration     0.7500    0.6000    0.6667         5
                  PortScan     0.9033    0.9501    0.9261     23757
               SSH-Patator     0.9739    0.9297    0.9513       882
  Web Attack - Brute Force     0.8889    0.3556    0.5079       225
Web Attack - Sql Injection     0.0267    0.6667

In [11]:
torch.save(model.state_dict(), '../models/cnn1d.pth')
print("Model saved.")

Model saved.


## CNN (1D-CNN) Results — Trained on SMOTE Resampled Data

### Overall Accuracy
CNN: 98.75% vs XGBoost: 99.89% vs Random Forest: 99.59%
Overall accuracy dropped slightly — but per-class results tell a better story.

### Comparison Table — Key Classes

| Class | RF Recall | XGB Recall | CNN Recall | Verdict |
|-------|-----------|------------|------------|---------|
| SQL Injection | 0.00 | 0.67 | 0.67 | Same as XGBoost |
| Web Attack XSS | 0.56 | 0.12 | 0.90 | CNN wins — big improvement |
| Bot | 0.98 | 0.67 | 0.88 | RF wins |
| Heartbleed | 1.00 | 0.50 | 1.00 | CNN matches RF |
| Brute Force | 0.73 | 0.97 | 0.36 | All three inconsistent |

### What CNN Fixed
XSS recall jumped from 0.12 (XGBoost) to 0.90 — the biggest single improvement across all three models so far. SMOTE combined with the CNN architecture clearly helped with this class.

### What CNN Still Gets Wrong
SQL Injection precision = 0.027. CNN catches 67% of SQL Injection attacks but predicts SQL Injection so broadly that 97% of those predictions are wrong — high false alarm rate. Brute Force recall dropped to 0.36, worse than both baselines.

### Core Insight
Each model fixes some rare classes while breaking others. No model handles all minority classes consistently at the same time. This is the fundamental problem that standard architectures and data-level fixes cannot solve alone.

### Connection to IAA-Transformer
IAA-Transformer bakes class frequency directly into the attention formula — forcing the model to pay attention to ALL rare classes simultaneously during every forward pass, not just the ones it happens to get right by chance.